# 🟣 07 — Cosmos 3: The Generator (Creating Video, Sound & More)

> **Goal:** the *creative* half of Cosmos 3 — turn a sentence into a brand-new image,
> video, or even video **with synchronized sound**.

---

## Understanding → Creating

Notebook 06 took *video in → words out*. The generator runs the **opposite** direction:
*words in → video out*. It runs **in-process** with Diffusers (no server needed).

```mermaid
flowchart LR
    P["📝 "A robot navigates<br/>a warehouse""]:::user
    G["🟣 Cosmos3GeneratorModel<br/>(Diffusers, in-process)"]:::gen
    OUT([🎬 new video / 🖼 image / 🔊 video+sound]):::out
    P --> G --> OUT
    classDef user fill:#ECEFF1,stroke:#607D8B,stroke-width:2px,color:#263238
    classDef agent fill:#E3F2FD,stroke:#1976D2,stroke-width:2px,color:#0D47A1
    classDef reason fill:#E8F5E9,stroke:#388E3C,stroke-width:2px,color:#1B5E20
    classDef gen fill:#F3E5F5,stroke:#8E24AA,stroke-width:2px,color:#4A148C
    classDef tool fill:#FFF3E0,stroke:#F57C00,stroke-width:2px,color:#E65100
    classDef data fill:#FFFDE7,stroke:#FBC02D,stroke-width:2px,color:#F57F17
    classDef out fill:#E0F2F1,stroke:#00897B,stroke-width:2px,color:#004D40
    classDef think fill:#FCE4EC,stroke:#D81B60,stroke-width:2px,color:#880E4F
```

!!! warning "Memory note"
    The reasoner (notebook 06) and this generator each load a 16B model. On a single
    ~46 GB GPU, **stop the reasoner server before running the generator**.

!!! info "One-time setup"
    ```bash
    just c3-setup-gen     # diffusers + cosmos_guardrail
    ```

In [ ]:
# 🔧 Preflight for the Generator — needs a big GPU; runs in-process (no server).
import os, sys, time
sys.path.insert(0, os.path.abspath(".."))

# 🔒 Cosmos tools confine file access to a workspace allow-list for safety.
# The notebooks live in notebooks/ but our sample media sits one level up, so we
# explicitly widen the workspace to the project root + /tmp. (This is how you grant
# the tools access to a folder — never wider than you need.)
os.environ["COSMOS_WORKSPACE"] = os.pathsep.join([os.path.abspath(".."), "/tmp"])
OUT_DIR = "/tmp/cosmos3_nb"
os.makedirs(OUT_DIR, exist_ok=True)

def gpu_ready():
    try:
        import torch; return torch.cuda.is_available()
    except Exception:
        return False

def big_enough(min_gb=40):
    try:
        import torch
        if not torch.cuda.is_available(): return False
        return torch.cuda.get_device_properties(0).total_memory/1e9 >= min_gb
    except Exception:
        return False

GPU = gpu_ready()
BIG = big_enough()
print("🎮 GPU available     :", GPU)
print("💾 ≥40GB VRAM (gen)  :", BIG)
if GPU and not BIG:
    print("   ↳ The 16B generator needs a large GPU. Cells will skip on smaller cards.")
print("📂 Output folder     :", OUT_DIR)

## Step 1 — Load the generator
One object handles every generation mode. First load also fetches weights.

In [ ]:
gen = None
if BIG:
    try:
        from strands_cosmos import Cosmos3GeneratorModel
        t0 = time.time()
        gen = Cosmos3GeneratorModel(model_id="nvidia/Cosmos3-Nano")
        print(f"✅ Generator ready in {time.time()-t0:.1f}s")
    except ImportError as e:
        print("⏭  Generator deps not in this env. Build them with: just c3-setup-gen")
        print(f"   ({str(e)[:120]})")
else:
    print("⏭  Need a large GPU (~46GB). Read along — the mode table still teaches the API.")

def safe_generate(**kw):
    """Run a generation, but never crash the notebook if gen deps/server are missing."""
    if gen is None:
        print("⏭  Generator not available — skipping."); return
    try:
        t0 = time.time()
        print(gen.generate(**kw))
        print(f"[{time.time()-t0:.1f}s]")
    except ImportError as e:
        print("⏭  Generator deps not in this env. Build them with: just c3-setup-gen")
        print(f"   ({str(e)[:120]})")
    except Exception as e:
        print(f"⏭  Skipped ({type(e).__name__}): {str(e)[:120]}")

## Step 2 — Text → Image (the fast one)
Start small: a single picture. Fewer steps = quicker.

In [ ]:
safe_generate(mode="text2image",
              prompt="A mobile robot in a warehouse aisle, photorealistic.",
              out_path=f"{OUT_DIR}/image.png",
              resolution="480", num_inference_steps=20)

## Step 3 — Text → Video
Now a short clip. `num_frames` × `fps` controls the length.

In [ ]:
safe_generate(mode="text2video",
              prompt="A robot navigates a warehouse and stops at a shelf.",
              out_path=f"{OUT_DIR}/video.mp4",
              resolution="480", num_frames=49, fps=16, num_inference_steps=25)

## Step 4 — Text → Video **with sound** 🔊
The showstopper: a video whose audio track is generated *in sync* (stereo AAC @ 48kHz).

In [ ]:
safe_generate(mode="text2video-with-sound",
              prompt="A robot pours water into a metal cup on a workbench.",
              out_path=f"{OUT_DIR}/video_sound.mp4",
              resolution="480", num_frames=49, fps=16,
              num_inference_steps=25, enable_sound=True)

## The full mode menu
Every generation mode uses the same `gen.generate(mode=...)` call:

| Mode | Input → Output | Tool equivalent |
|------|----------------|-----------------|
| `text2image` | text → 🖼 image | `cosmos3_text2image` |
| `text2video` | text → 🎬 video | `cosmos3_text2video` |
| `image2video` | 🖼 + text → 🎬 video | `cosmos3_image2video` |
| `text2video-with-sound` | text → 🎬🔊 video+audio | `cosmos3_text2video_sound` |
| `image2video-with-sound` | 🖼 + text → 🎬🔊 | `cosmos3_image2video_sound` |

Plus **world-model / robotics** tools: `cosmos3_forward_dynamics`,
`cosmos3_inverse_dynamics`, `cosmos3_policy` — see `examples/08_cosmos3_action.py`.

# 🎓 What you learned

| Concept | Takeaway |
|---------|----------|
| `Cosmos3GeneratorModel` | The **creative** half — runs in-process (Diffusers) |
| `gen.generate(mode=...)` | One call, many modes (image/video/sound) |
| `num_frames`, `fps`, steps | Control length, smoothness, and quality |
| Memory | Don't run reasoner + generator together on one GPU |

## 🎉 You finished the tour!

You can now: reason over text 🟢, see video & images 👀, think step-by-step 🧠, drive robots
🤖, compose tools 🟠, and **generate** video + sound 🟣.

**Where to go next:** the [`examples/`](../examples) folder has runnable scripts for every
notebook (plus advanced training & action models), and the [docs](../docs) go deeper on each
topic.